# Chapter 5 — Beyond ID-Only Towers: What Features Buy You (and What They Cost)

Companion to Section 5.2.4. We compare three item-tower designs on the same data:

1. **ID-only** — the `TwoTower` baseline: every item is unique by construction, but a new item is invisible.
2. **Feature-only** — genres + release year: works for brand-new items, but two movies with the same features get the **same embedding**, no matter how differently users treat them.
3. **ID + features** — the design from Figure 5.5: the ID restores resolution for items with history; the features carry brand-new items.

To make the cold-start difference measurable, we hold a sample of items **completely out of training** and evaluate retrieval recall separately on warm and cold items.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "recsys").exists():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print(f"Project root: {project_root}")

In [ ]:
import random
import re

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from recsys.data.loaders import load_movielens
from recsys.data.preprocessing import (
    add_item_idx, build_item_index, filter_min_item_ratings, filter_positive,
    sample_active_users, temporal_split_per_user, user_item_lists,
)
from recsys.fourstage_recsys.retrieval.two_tower import (
    TwoTower, create_negative_pairs, create_positive_pairs,
    extract_embeddings, train_bce,
)
from recsys.evaluation.metrics import recall_at_k

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

## 1. Data — same preparation as `01_similarity_learning`

In [ ]:
ratings, movies = load_movielens("ml-25m", data_dir=project_root / "data")

ratings = sample_active_users(ratings, n_users=10_000, min_ratings=20, seed=SEED)
ratings = filter_min_item_ratings(ratings, min_ratings=10)
interactions = filter_positive(ratings, threshold=4.0)

item_ids, item_to_idx, idx_to_title, idx_to_genres = build_item_index(
    interactions, movies)
interactions = add_item_idx(interactions, item_to_idx)
num_items = len(item_ids)

train_df, test_df = temporal_split_per_user(interactions, test_frac=0.2)
relevance_sets = test_df.groupby("userId")["item_idx"].apply(set).to_dict()

print(f"{interactions.userId.nunique():,} users, {num_items:,} movies")

## 2. Item features — and how coarse they are

Genre multi-hot plus release year, the features named in Section 5.2.4. Before training anything, we measure the feature space's **resolution**: how many movies share their exact (genres, year) signature with at least one other movie. Every such group is an equivalence class a feature-only tower cannot split.

In [ ]:
def build_item_features(idx_to_title: dict, idx_to_genres: dict,
                        num_items: int):
    """Genre multi-hot + scaled release year, one row per item index."""
    genre_vocab = sorted({g for gs in idx_to_genres.values() for g in gs
                          if g != "(no genres listed)"})
    g_pos = {g: j for j, g in enumerate(genre_vocab)}
    feats = np.zeros((num_items, len(genre_vocab) + 1), dtype=np.float32)
    years = np.full(num_items, np.nan)
    for i in range(num_items):
        for g in idx_to_genres.get(i, []):
            if g in g_pos:
                feats[i, g_pos[g]] = 1.0
        m = re.search(r"\((\d{4})\)\s*$", idx_to_title.get(i, ""))
        if m:
            years[i] = int(m.group(1))
    years = np.where(np.isnan(years), np.nanmean(years), years)
    feats[:, -1] = (years - 1900.0) / 100.0
    return feats, genre_vocab


item_features, genre_vocab = build_item_features(
    idx_to_title, idx_to_genres, num_items)
print(f"Feature dim: {item_features.shape[1]} "
      f"({len(genre_vocab)} genres + year)")

# Resolution of the feature space: exact-signature equivalence classes
signatures = [tuple(row) for row in item_features]
sig_counts = pd.Series(signatures).value_counts()
collided = int(sig_counts[sig_counts > 1].sum())
print(f"Distinct signatures: {len(sig_counts):,} for {num_items:,} movies")
print(f"Movies sharing their signature with at least one other: "
      f"{collided:,} ({collided / num_items:.0%})")
print(f"Largest equivalence class: {int(sig_counts.iloc[0])} movies")

## 3. Cold-item protocol

We sample 5% of items (weighted toward items that appear in the test set) and delete **every training interaction** involving them. All three models keep these items in their index space, but none ever sees them in a training pair — exactly the situation of an article published five minutes ago. The ID-only model's embeddings for them are untrained noise; the feature towers can still compute them from metadata.

In [ ]:
rng = np.random.default_rng(SEED)
test_items = sorted({i for s in relevance_sets.values() for i in s})
cold_items = set(rng.choice(test_items,
                            size=max(1, int(0.05 * num_items)),
                            replace=False).tolist())
warm_items = set(range(num_items)) - cold_items

train_df_warm = train_df[~train_df["item_idx"].isin(cold_items)]
train_items_idx = user_item_lists(train_df_warm, item_col="item_idx")

positive_pairs = create_positive_pairs(train_items_idx, max_pairs_per_user=50)
negative_pairs = create_negative_pairs(positive_pairs, num_items,
                                       num_neg_per_pos=5)
touched = {i for p in positive_pairs for i in p}
assert not (touched & cold_items), "cold items leaked into training pairs"
print(f"{len(cold_items):,} cold items held out; "
      f"{len(positive_pairs):,} positive pairs from warm items only")

## 4. The feature-rich tower

Same two-tower shape as Listing 5.3 — only the input changes. With `use_id=True`, the ID embedding is **one feature among several**, not replaced by the metadata (Figure 5.5); with `use_id=False`, the tower sees metadata alone. The class implements the same interface as `TwoTower` (`forward`, `loss`, `query_table`, `candidate_table`), so the package's `train_bce` and `extract_embeddings` work unchanged.

In [ ]:
class FeatureTwoTower(nn.Module):
    """Two-tower model whose towers read item features, optionally + ID.

    ``id_dropout`` randomly zeroes the whole ID embedding for a fraction of
    training rows, so a zeroed ID is *in-distribution* — which is what makes
    masking untrained IDs at inference safe for brand-new items.
    """

    def __init__(self, num_items: int, item_features: np.ndarray,
                 emb_dim: int = 64, id_dim: int = 32, use_id: bool = True,
                 id_dropout: float = 0.0, c_vector: float = 1e-6):
        super().__init__()
        self.use_id = use_id
        self.id_dropout = id_dropout
        self.register_buffer(
            "features", torch.tensor(item_features, dtype=torch.float32))
        in_dim = item_features.shape[1] + (id_dim if use_id else 0)
        if use_id:
            self.embedding1 = nn.Embedding(num_items, id_dim)
            self.embedding2 = nn.Embedding(num_items, id_dim)
        self.tower_one = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, emb_dim))
        self.tower_two = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Linear(128, emb_dim))
        self.bce = nn.BCEWithLogitsLoss()
        self.c_vector = c_vector

    def _inputs(self, items, table, mask_id=None):
        x = self.features[items]
        if not self.use_id:
            return x
        idv = table(items)
        if self.training and self.id_dropout > 0:            # in-distribution zeros
            keep = (torch.rand(idv.shape[0], 1, device=idv.device)
                    > self.id_dropout).float()
            idv = idv * keep
        if mask_id is not None:                              # untrained IDs off
            idv = idv * (~mask_id).float().unsqueeze(-1)
        return torch.cat([idv, x], dim=-1)

    def forward(self, item_1, item_2):
        e1 = self.tower_one(self._inputs(item_1, getattr(self, "embedding1", None)))
        e2 = self.tower_two(self._inputs(item_2, getattr(self, "embedding2", None)))
        return torch.sum(e1 * e2, dim=1)

    def loss(self, pred_logits, label):
        bce = self.bce(pred_logits, label)
        if not self.use_id:
            return bce
        reg = sum(torch.sum(p ** 2.0) for p in
                  [self.embedding1.weight, self.embedding2.weight])
        return bce + reg * self.c_vector

    def query_table(self):
        idx = torch.arange(self.features.shape[0], device=self.features.device)
        return self.tower_one(self._inputs(idx, getattr(self, "embedding1", None)))

    def candidate_table(self, cold: set | None = None):
        idx = torch.arange(self.features.shape[0], device=self.features.device)
        mask = None
        if cold is not None and self.use_id:
            mask = torch.zeros(len(idx), dtype=torch.bool,
                               device=self.features.device)
            mask[list(cold)] = True
        return self.tower_two(self._inputs(
            idx, getattr(self, "embedding2", None), mask_id=mask))

    @torch.no_grad()
    def candidate_embeddings(self, cold: set | None = None) -> np.ndarray:
        """Normalized candidate table — normalize here, or cold items lose
        every dot product to larger-norm warm rows and fail *silently*
        (the same trap as Section 5.3.2's METRIC_L2 warning)."""
        self.eval()
        cand = self.candidate_table(cold=cold).cpu().numpy().astype(np.float32)
        return cand / np.clip(
            np.linalg.norm(cand, axis=1, keepdims=True), 1e-12, None)

In [ ]:
EMB_DIM = 64
EPOCHS = 6

models = {
    "ID-only": TwoTower(num_items, emb_dim=EMB_DIM),
    "Feature-only": FeatureTwoTower(num_items, item_features,
                                    emb_dim=EMB_DIM, use_id=False),
    "ID + features (naive)": FeatureTwoTower(num_items, item_features,
                                             emb_dim=EMB_DIM, use_id=True),
    "ID + features (dropout + cold mask)": FeatureTwoTower(
        num_items, item_features, emb_dim=EMB_DIM, use_id=True,
        id_dropout=0.3),
}
cand_tables = {}
for name, model in models.items():
    print(f"--- {name} ---")
    model, _ = train_bce(model, positive_pairs, negative_pairs,
                         epochs=EPOCHS, batch_size=2048, device=DEVICE)
    if isinstance(model, FeatureTwoTower):
        cold = cold_items if "cold mask" in name else None
        cand_tables[name] = model.candidate_embeddings(cold=cold)
    else:
        _, cand_tables[name] = extract_embeddings(model)

## 5. The collision, made concrete

The feature-only tower is a function of the features and nothing else, so movies with identical signatures **must** land on the same point. We verify it, then look at what it does to a spot check.

In [ ]:
big_sig = sig_counts.index[0]
cls = [i for i, s in enumerate(signatures) if s == big_sig][:12]
for name in ["Feature-only", "ID + features (naive)"]:
    cand = cand_tables[name]
    spread = float(np.max(np.linalg.norm(
        cand[cls] - cand[cls[0]], axis=1)))
    print(f"{name}: max distance within a {len(cls)}-movie "
          f"identical-signature class = {spread:.4f}")
print("Movies in that class:",
      [idx_to_title[i] for i in cls[:4]], "...")

## 6. Warm vs. cold retrieval recall

Overall averages hide the failure mode (compare Figure 5.9), so we stratify: for each user, recall over their relevant **warm** items and their relevant **cold** items separately. The user query is their history's mean embedding, history items are masked, and cold users without cold relevant items simply don't contribute to the cold average.

In [ ]:
def stratified_cold_recall(query_vecs, cand_vecs, train_items_by_user,
                           relevance_sets, cold_items, k=100):
    warm_r, cold_r = [], []
    for user_id, relevant in relevance_sets.items():
        history = train_items_by_user.get(user_id)
        if not history:
            continue
        q = cand_vecs[history].mean(axis=0)
        q = q / max(np.linalg.norm(q), 1e-12)
        scores = cand_vecs @ q
        scores[history] = -np.inf
        ranked = np.argsort(-scores)[:k].tolist()
        rel_warm = relevant - cold_items
        rel_cold = relevant & cold_items
        if rel_warm:
            warm_r.append(recall_at_k(ranked, rel_warm, k))
        if rel_cold:
            cold_r.append(recall_at_k(ranked, rel_cold, k))
    return {"warm Recall@100": float(np.mean(warm_r)),
            "cold Recall@100": float(np.mean(cold_r)),
            "users (warm/cold)": f"{len(warm_r)}/{len(cold_r)}"}


results = pd.DataFrame({
    name: stratified_cold_recall(cand, cand, train_items_idx,
                                 relevance_sets, cold_items)
    for name, cand in cand_tables.items()
}).T
results

## 7. Spot check: a warm anchor under each design

In [ ]:
def neighbors(cand_vecs, anchor, k=5):
    v = cand_vecs / np.clip(
        np.linalg.norm(cand_vecs, axis=1, keepdims=True), 1e-12, None)
    scores = v @ v[anchor]
    top = [i for i in np.argsort(-scores) if i != anchor][:k]
    return [f"{scores[i]:.3f}  {idx_to_title[i]}" for i in top]


anchor = next(i for i, t in idx_to_title.items()
              if "Toy Story (1995)" in t)
for name, cand in cand_tables.items():
    print(f"--- {idx_to_title[anchor]} — {name} ---")
    print("\n".join(neighbors(cand, anchor)))
    print()

## What the results show

- **The collision is exact.** Within an identical-signature class, the feature-only spread is 0.0000 — identical inputs, identical embeddings — and its "nearest neighbors" for any anchor are its signature twins at similarity 1.000. The ID variants separate the same class. This is the resolution cap from Section 5.2.4, measured.
- **ID-only is blind to cold items.** Its cold-item embeddings are untrained noise; cold recall sits at chance level, and at a production catalog size (k=100 of millions) chance is effectively zero.
- **Naive concatenation does *not* degrade gracefully.** The trained tower learns to lean on the ID — it is fully informative for every warm item — so a cold item's untrained ID corrupts the output *more* than having no ID at all. In our runs, naive ID + features had *worse* cold recall than ID-only.
- **The fix is a recipe, not a checkbox**: (1) train with **ID dropout**, so a zeroed ID is in-distribution; (2) **mask the ID at inference** for items with no interaction history; (3) **normalize** the resulting table — masked cold rows come out with smaller norms, and without normalization they lose every dot product to warm rows and vanish from retrieval *silently*, the same failure family as the METRIC_L2 trap.

The trade-off in one line: features set the floor (no item is invisible) and the ID sets the ceiling (no two items are forced together) — **but only if the serving path knows when to trust the ID**. That is the honest version of Figure 5.5's design, and the honest caveat for `warm_start_embedding`: metadata places a new item in the right region at the feature space's precision, not the catalog's.